# Baseline models: predicting wine color from reviews

Use the stratified train and validation sets saved by `data_split.ipynb`. Evaluate all five colors in `included_types`. Models use only `description` and review-length features derived from it; `variety` is reserved for error inspection. Keep `test.parquet` untouched until the final model is chosen.

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

In [2]:
included_types = ['red', 'white', 'sparkling', 'rose', 'dessert']
train = pd.read_parquet('train.parquet')
validation = pd.read_parquet('validation.parquet')
train = train.loc[train['color'].isin(included_types)].copy()
validation = validation.loc[validation['color'].isin(included_types)].copy()
assert set(included_types) <= set(train['color'])
assert set(included_types) <= set(validation['color'])
assert train.index.intersection(validation.index).empty, 'Train and validation rows overlap.'
assert train['description'].notna().all() and validation['description'].notna().all()

wine = pd.concat([train, validation])  # Exploratory summaries only; never fit on this combined frame.
pd.DataFrame({
    'training': train['color'].value_counts(),
    'validation': validation['color'].value_counts(),
}).reindex(included_types)

,training,validation
color,,
red,62291,20764
white,32550,10850
sparkling,3322,1107
rose,2471,823
dessert,872,291


## Review length and data size

Check lengths before vectorizing. The count matrix will be sparse, so memory scales with the number of words actually present in each review rather than rows × vocabulary size.

## Review length features

Count alphabetic words in each description and measure their mean and maximum length. Punctuation and numbers are excluded. Descriptions with no alphabetic words receive zeros. These features come only from `description`.


In [3]:
import re

word_pattern = re.compile(r'(?u)\b[^\W\d_]+\b')
length_feature_cols = ['review_word_count', 'mean_word_length', 'max_word_length']

def review_length_features(description):
    lengths = [len(word) for word in word_pattern.findall(description)]
    if not lengths:
        return 0, 0.0, 0
    return len(lengths), sum(lengths) / len(lengths), max(lengths)

for frame in (train, validation):
    frame[length_feature_cols] = pd.DataFrame(
        [review_length_features(text) for text in frame['description'].fillna('')],
        index=frame.index,
        columns=length_feature_cols,
    )
wine = pd.concat([train, validation])
train[length_feature_cols].describe().round(1)

,review_word_count,mean_word_length,max_word_length
count,101506.0,101506.0,101506.0
mean,41.1,4.7,10.5
std,11.5,0.4,1.5
min,3.0,2.8,4.0
25%,33.0,4.5,10.0
50%,40.0,4.7,10.0
75%,48.0,5.0,12.0
max,136.0,6.8,20.0


In [4]:
review_chars = wine['description'].str.len()
print(f'{len(wine):,} reviews; {wine["description"].nunique():,} distinct descriptions')
print(f'Raw description text: {wine["description"].memory_usage(deep=True) / 2**20:,.1f} MiB')
pd.DataFrame({'words': wine['review_word_count'], 'characters': review_chars}).describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(1)

135,341 reviews; 135,341 distinct descriptions
Raw description text: 33.4 MiB


,words,characters
count,135341.0,135341.0
mean,41.1,242.3
std,11.5,67.9
min,3.0,17.0
50%,40.0,237.0
90%,56.0,330.0
95%,61.0,361.0
99%,73.0,430.0
max,136.0,829.0


In [5]:
wine.columns

Index(['country', 'description', 'designation', 'points', 'price', 'province',
       'region_1', 'region_2', 'variety', 'winery', 'taster_name',
       'taster_twitter_handle', 'title', 'color', 'review_word_count',
       'mean_word_length', 'max_word_length'],
      dtype='str')

## Use the saved split

`data_split.ipynb` already made a stratified 60/20/20 train/validation/test split. The notebook uses its saved training and validation rows, including all colors in `included_types`. Fit the vocabulary and feature transformations on training reviews only. Validation measures generalization during model comparison; the test split is reserved for final evaluation.

In [6]:
y_train = train['color']
y_val = validation['color']
print(f'Train: {len(train):,}; validation: {len(validation):,}')
print('Training class proportions:')
print(y_train.value_counts(normalize=True).reindex(included_types).round(3))

Train: 101,506; validation: 33,835
Training class proportions:
color
red          0.614
white        0.321
sparkling    0.033
rose         0.024
dessert      0.009
Name: proportion, dtype: float64


## Unigram and bigram bag of words

`CountVectorizer` counts individual words and adjacent word pairs. `min_df=5` omits very rare terms, and `max_features=50_000` caps the vocabulary. The tokenizer keeps alphabetic words of at least two characters. The vocabulary is fit on training reviews and reused for validation reviews.

In [7]:
vectorizer = CountVectorizer(ngram_range=(1, 2), min_df=5, max_features=50_000, token_pattern=r'(?u)\b[^\W\d_]{2,}\b', dtype=np.int32)
X_train_text = vectorizer.fit_transform(train['description'].fillna(''))
X_val_text = vectorizer.transform(validation['description'].fillna(''))

def sparse_mib(matrix):
    return (matrix.data.nbytes + matrix.indices.nbytes + matrix.indptr.nbytes) / 2**20

print(f'Vocabulary: {len(vectorizer.vocabulary_):,} unigrams/bigrams')
print(f'Train matrix: {X_train_text.shape}, {X_train_text.nnz:,} nonzero counts, {sparse_mib(X_train_text):,.1f} MiB')
print(f'Validation matrix: {X_val_text.shape}, {X_val_text.nnz:,} nonzero counts, {sparse_mib(X_val_text):,.1f} MiB')
print('Sample features:', vectorizer.get_feature_names_out()[:20].tolist())

Vocabulary: 50,000 unigrams/bigrams
Train matrix: (101506, 50000), 6,326,665 nonzero counts, 48.7 MiB
Validation matrix: (33835, 50000), 2,097,050 nonzero counts, 16.1 MiB
Sample features: ['abacela', 'abbey', 'abbreviated', 'abbreviated finish', 'ability', 'ability to', 'able', 'able to', 'ably', 'abound', 'abound in', 'abound on', 'abound with', 'abounds', 'abounds in', 'abounds with', 'about', 'about all', 'about anything', 'about as']


## Other available predictors

The remaining columns can be added later. Numeric columns need imputation; categorical columns need missing-value handling and encoding. Check cardinality before one-hot encoding `winery`, `designation`, or `title`, since these may create large feature spaces. `title` and `designation` can also contain grape or color names, so decide whether those clues are allowed in the eventual prediction setting. `points` and `price` may be unavailable at prediction time.

In [8]:
other_columns = [c for c in wine.columns if c not in ['description', 'color', *length_feature_cols]]
pd.DataFrame({
    'dtype': wine[other_columns].dtypes.astype(str),
    'missing': wine[other_columns].isna().sum(),
    'unique_non_null': wine[other_columns].nunique(dropna=True),
}).sort_values('unique_non_null', ascending=False)

,dtype,missing,unique_non_null
title,str,39423,95115
designation,str,40011,40760
winery,str,0,18066
region_1,str,22400,1298
variety,str,0,706
province,str,52,473
price,float64,10165,385
country,str,52,49
points,int64,0,21
taster_name,str,59355,19


## Description-only classifiers and bias–variance comparison

A majority-class dummy model gives a reference for this imbalanced five-class problem. Compare it with multinomial Naive Bayes, logistic regression, and a linear SVM. All three text classifiers use unigram/bigram counts and three length features; logistic regression and SVM use TF-IDF weights. Fit scaling and TF-IDF on training data only.

Compare **training versus validation macro F1** and balanced accuracy. Low scores on both splits suggest high bias; a strong training score with a much lower validation score suggests high variance. These are diagnostic clues, not formal estimates of statistical bias or variance. Macro F1 gives rare colors equal weight, while accuracy can hide poor minority-class performance.

In [9]:
from scipy.sparse import csr_matrix, hstack
from sklearn.preprocessing import MaxAbsScaler
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report, confusion_matrix

length_scaler = MaxAbsScaler()
X_train_length = csr_matrix(length_scaler.fit_transform(train[length_feature_cols]))
X_val_length = csr_matrix(length_scaler.transform(validation[length_feature_cols]))

tfidf = TfidfTransformer(sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_val_tfidf = tfidf.transform(X_val_text)

X_train_counts_with_length = hstack([X_train_text, X_train_length], format='csr')
X_val_counts_with_length = hstack([X_val_text, X_val_length], format='csr')
X_train_tfidf_with_length = hstack([X_train_tfidf, X_train_length], format='csr')
X_val_tfidf_with_length = hstack([X_val_tfidf, X_val_length], format='csr')
print(f'Train matrix with review length features: {X_train_tfidf_with_length.shape}, {sparse_mib(X_train_tfidf_with_length):,.1f} MiB')

Train matrix with review length features: (101506, 50003), 76.3 MiB


In [10]:
models = {
    'Majority class': (DummyClassifier(strategy='most_frequent'), X_train_counts_with_length, X_val_counts_with_length),
    'Multinomial Naive Bayes': (MultinomialNB(alpha=1.0), X_train_counts_with_length, X_val_counts_with_length),
    'Logistic regression': (LogisticRegression(max_iter=1000, solver='lbfgs'), X_train_tfidf_with_length, X_val_tfidf_with_length),
    'Linear SVM': (LinearSVC(max_iter=5000, dual=False), X_train_tfidf_with_length, X_val_tfidf_with_length),
}

score_rows = []
predictions = {}
for name, (model, X_fit, X_eval) in models.items():
    model.fit(X_fit, y_train)
    for split, X, y in [('training', X_fit, y_train), ('validation', X_eval, y_val)]:
        y_pred = model.predict(X)
        if split == 'validation':
            predictions[name] = y_pred
        score_rows.append({
            'model': name,
            'split': split,
            'accuracy': accuracy_score(y, y_pred),
            'balanced_accuracy': balanced_accuracy_score(y, y_pred),
            'macro_f1': f1_score(y, y_pred, labels=included_types, average='macro', zero_division=0),
        })
    print(f'Finished: {name}')

scores = pd.DataFrame(score_rows)
results_df = scores.pivot(index='model', columns='split', values=['accuracy', 'balanced_accuracy', 'macro_f1'])
results_df.columns = [f'{metric}_{split}' for metric, split in results_df.columns]
results_df['macro_f1_gap'] = results_df['macro_f1_training'] - results_df['macro_f1_validation']
results_df['balanced_accuracy_gap'] = results_df['balanced_accuracy_training'] - results_df['balanced_accuracy_validation']
results_df = results_df.sort_values('macro_f1_validation', ascending=False)
print(results_df.round(3).to_string())

Finished: Majority class
Finished: Multinomial Naive Bayes
Finished: Logistic regression
Finished: Linear SVM
                         accuracy_training  accuracy_validation  balanced_accuracy_training  balanced_accuracy_validation  macro_f1_training  macro_f1_validation  macro_f1_gap  balanced_accuracy_gap
model                                                                                                                                                                                                 
Linear SVM                           0.993                0.950                       0.946                         0.677              0.966                0.737         0.228                  0.269
Multinomial Naive Bayes              0.947                0.932                       0.841                         0.724              0.791                0.699         0.092                  0.116
Logistic regression                  0.963                0.945                       0.695   

## Per-class validation errors

Inspect precision and recall for every color. The confusion-matrix rows are actual colors and columns are predicted colors. This reveals whether a good overall score hides weak performance on `rose`, `dessert`, or `sparkling`.

In [11]:
for name, y_pred in predictions.items():
    print(f'\n{name}')
    print(classification_report(y_val, y_pred, labels=included_types, zero_division=0))
    print(pd.DataFrame(
        confusion_matrix(y_val, y_pred, labels=included_types),
        index=[f'actual {color}' for color in included_types],
        columns=[f'predicted {color}' for color in included_types],
    ))


Majority class
              precision    recall  f1-score   support

         red       0.61      1.00      0.76     20764
       white       0.00      0.00      0.00     10850
   sparkling       0.00      0.00      0.00      1107
        rose       0.00      0.00      0.00       823
     dessert       0.00      0.00      0.00       291

    accuracy                           0.61     33835
   macro avg       0.12      0.20      0.15     33835
weighted avg       0.38      0.61      0.47     33835

                  predicted red  predicted white  predicted sparkling  \
actual red                20764                0                    0   
actual white              10850                0                    0   
actual sparkling           1107                0                    0   
actual rose                 823                0                    0   
actual dessert              291                0                    0   

                  predicted rose  predicted dessert  
ac

## Misclassified validation reviews

Inspect errors from the model with the highest validation macro F1. `variety` appears only in this error table and is never a feature. The original row index is retained for lookup.

In [12]:
model_to_inspect = results_df.index[0]  # Or choose a name from predictions.
y_pred = predictions[model_to_inspect]
wrong = y_pred != y_val.to_numpy()

error_df = validation.loc[y_val.index[wrong]].copy()
error_df = error_df.rename(columns={'color': 'actual_color'})
error_df.insert(1, 'predicted_color', y_pred[wrong])
first_cols = ['actual_color', 'predicted_color', 'variety', 'description', *length_feature_cols]
error_df = error_df[first_cols + [col for col in error_df.columns if col not in first_cols]]
print(f'{model_to_inspect}: {len(error_df):,} incorrect predictions out of {len(validation):,} validation reviews')
error_df

Linear SVM: 1,675 incorrect predictions out of 33,835 validation reviews


,actual_color,predicted_color,variety,description,review_word_count,mean_word_length,max_word_length,country,designation,points,price,province,region_1,region_2,winery,taster_name,taster_twitter_handle,title
47694,white,red,Sauvignon Blanc,Future vintages of this wine will bear the new...,54,4.685185,11,US,Klipsun Vineyard,90,8.0,Washington,Columbia Valley (WA),Columbia Valley,Waterbrook,NaN,NaN,NaN
45475,white,red,Greco,"As evidenced by its solid structure, this is a...",33,4.545455,9,Italy,NaN,83,20.0,Southern Italy,Greco di Tufo,NaN,Feudi di San Gregorio,NaN,NaN,NaN
80285,sparkling,white,Champagne Blend,"Fermented in oak, this wine with its crisp app...",41,4.170732,9,France,Millésime Brut,91,40.0,Champagne,Champagne,NaN,Louise Brison,Roger Voss,@vossroger,Louise Brison 2010 Millésime Brut (Champagne)
50286,red,white,Red Blend,"It's soft, slightly oxidized, and generic. It'...",20,4.750000,10,US,Rough House Red,85,30.0,Washington,Walla Walla Valley (WA),Columbia Valley,Covington,Paul Gregutt,@paulgwine,Covington 2006 Rough House Red Red (Walla Wall...
63739,white,red,Chardonnay,"A négociant wine, sharp and spicy, with vivid ...",47,4.574468,9,US,NaN,86,18.0,Washington,Washington,Washington Other,Hedgeline,Paul Gregutt,@paulgwine,Hedgeline 2009 Chardonnay (Washington)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23036,sparkling,white,Prosecco,There's the slightest trace of oxidation on th...,38,5.026316,14,Italy,Riva dei Fiori Brut,82,21.0,Veneto,Prosecco di Conegliano e Valdobbiadene,NaN,Montesel Renzo,NaN,NaN,NaN
117318,dessert,red,Port,"Made in the style of a classic Port, this powe...",59,4.847458,11,US,Opportunity,94,35.0,California,Livermore Valley,Central Coast,Occasio,Jim Gordon,@gordone_cellars,Occasio NV Opportunity Port (Livermore Valley)
99336,red,rose,Bordeaux-style Red Blend,With an edge of tannin and with sweet strawber...,40,4.500000,10,France,NaN,86,17.0,Bordeaux,Bordeaux Clairet,NaN,Château Vignol,Roger Voss,@vossroger,Château Vignol 2016 Bordeaux Clairet
70273,sparkling,red,Lambrusco,"Frizzy and fun, this red semi-sparkler offers ...",31,4.451613,8,Italy,NaN,82,15.0,Central Italy,Lambrusco Reggiano,NaN,Moro di Rinaldini,NaN,NaN,Moro di Rinaldini 2011 Lambrusco Reggiano


In [13]:
error_df['actual_color'].value_counts().reindex(included_types, fill_value=0).to_frame('validation_errors')

,validation_errors
actual_color,
red,319
white,303
sparkling,485
rose,368
dessert,200


In [14]:
error_df['variety'].value_counts().head(20)

variety
Rosé                          341
Sparkling Blend               219
Champagne Blend               157
Port                          133
Chardonnay                    113
Pinot Noir                     88
Prosecco                       46
Moscato                        45
Bordeaux-style Red Blend       28
Red Blend                      23
Glera                          22
Sauvignon Blanc                17
Riesling                       17
Rosato                         16
Moscatel                       16
Bordeaux-style White Blend     14
White Blend                    13
Viognier                       13
Pinot Grigio                   13
Cabernet Sauvignon             13
Name: count, dtype: int64

In [15]:
error_df[['actual_color', 'predicted_color', 'variety', 'description']].head(10)

,actual_color,predicted_color,variety,description
47694,white,red,Sauvignon Blanc,Future vintages of this wine will bear the new...
45475,white,red,Greco,"As evidenced by its solid structure, this is a..."
80285,sparkling,white,Champagne Blend,"Fermented in oak, this wine with its crisp app..."
50286,red,white,Red Blend,"It's soft, slightly oxidized, and generic. It'..."
63739,white,red,Chardonnay,"A négociant wine, sharp and spicy, with vivid ..."
98882,dessert,red,Port,Get past the meaty aromas and the wine is smoo...
52628,rose,red,Rosé,"A fresh, high-acid wine, with flavors of raspb..."
121157,rose,white,Rosado,A cloudy rusted color and soupy aromas render ...
83086,sparkling,white,Sparkling Blend,"Snappy and a little green on the nose, with hi..."
166952,rose,red,Rosé,Intense mineral tones of smoke and crushed sla...


In [16]:
# Compare the score gap with the validation score before choosing the next experiment.
diagnostic = results_df[['macro_f1_training', 'macro_f1_validation', 'macro_f1_gap',
                         'balanced_accuracy_training', 'balanced_accuracy_validation',
                         'balanced_accuracy_gap']].round(3)
print('A large positive gap suggests high variance; low scores on both splits suggest high bias.')
print('Next: test stronger regularization or more training data for models with large gaps,')
print('and class weighting or improved features for colors with low validation recall.')
print(diagnostic.to_string())

A large positive gap suggests high variance; low scores on both splits suggest high bias.
Next: test stronger regularization or more training data for models with large gaps,
and class weighting or improved features for colors with low validation recall.
                         macro_f1_training  macro_f1_validation  macro_f1_gap  balanced_accuracy_training  balanced_accuracy_validation  balanced_accuracy_gap
model                                                                                                                                                         
Linear SVM                           0.966                0.737         0.228                       0.946                         0.677                  0.269
Multinomial Naive Bayes              0.791                0.699         0.092                       0.841                         0.724                  0.116
Logistic regression                  0.764                0.675         0.089                       0.695    